## BEIR

In [14]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

dataset = "scifact"
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
data_path = util.download_and_unzip(url, "datasets")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

100%|██████████| 5183/5183 [00:00<00:00, 207338.98it/s]


In [15]:
c_ids = list(corpus.keys())
c_text = [
    f"{corpus[doc_id]["title"].strip()} {corpus[doc_id]["text"].strip()}".strip()
    for doc_id in c_ids
    ]
c_text[0]

'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficient

## SentenceTransformers

In [16]:
from sentence_transformers import SentenceTransformer

# Load pretrained model from Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)

# Calculate embedding similarities
similarities = model.similarity(embeddings, embeddings)
# We should expect to see perfect similarity on the diagonal
print(similarities)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12986.63it/s]


(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


## FAISS

In [17]:
import numpy as np
d = 64                  # dimensionl
nb = 100000             # database size
nq = 10000              # nb of queries
np.random.seed(1234)    # make reproducible
xb = np.random.random((nb, d)).astype('float32') # random tensor of shape [nb, d] in [0, 1)
xb[:, 0] += np.arange(nb) / 1000.
xq = np.random.random((nq, d)).astype('float32')
xq[:, 0] += np.arange(nq) / 1000.
# The np.arange lines just introduce some correlation between q and corpus to test faiss as opposed to pure random noise

In [18]:
import faiss
index = faiss.IndexFlatL2(d)
print(index.is_trained)
index.add(xb)
print(index.ntotal)

True
100000


In [19]:
# Searching:
k = 4                           # get 4 closest neighbours
D, I = index.search(xb[:5], k)  # sanity check
print(I)                        # The first element of the 5 rows should be its corresponding row pos - i.e. the most similar element should be itself.
print(D)
D, I = index.search(xq, k)

[[  0 393 363  78]
 [  1 555 277 364]
 [  2 304 101  13]
 [  3 173  18 182]
 [  4 288 370 531]]
[[1.1444092e-05 7.1751823e+00 7.2076302e+00 7.2511673e+00]
 [0.0000000e+00 6.3235626e+00 6.6845779e+00 6.7999382e+00]
 [3.8146973e-06 5.7964058e+00 6.3917274e+00 7.2815208e+00]
 [3.8146973e-06 7.2779064e+00 7.5279846e+00 7.6628418e+00]
 [0.0000000e+00 6.7638016e+00 7.2951202e+00 7.3688164e+00]]
